In [1]:
import json
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.optuna_objective import create_objective
from src.utils.telegram import send_message

### Config

In [2]:
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "cb"
    data_id: str = "057"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 1
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Initial params
    use_initial: Literal["never", "manual"] = "never"
    initial_param_sources: list[tuple[str, int]] = field(default_factory=list)   # (study_name, n_trial) 例: ("xgb-001", 1)

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"

    # Option
    opts: dict = field(default_factory=dict)


cfg = Config()
cfg.initial_param_sources = [("lgbm-057", 16)]

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Build & Run

In [ ]:
# --- helper: initial params loader ---
def load_initial_params(sources: list[tuple[str, int]]) -> list[dict]:
    loaded = []
    for study, n_trial in sources:
        path = Path(f"../../artifacts/optuna/{study}/trl{n_trial}.json")
        with path.open("r") as f:
            params = json.load(f)["params"]
        loaded.append(params)
    return loaded


# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


objective = create_objective(
    cfg.model_name,
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=cfg.opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

initial_params = None
if cfg.use_initial == "manual":
    initial_params = load_initial_params(cfg.initial_param_sources)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner,
    initial_params=initial_params
)

[I 2025-10-07 13:24:50,733] Using an existing study with name 'cb-057' instead of creating a new one.


[initial] none


  0%|          | 0/1 [00:00<?, ?it/s]

Fold Col: 5fold-s42
Free CPU Mem: 16.96 GB
Free GPU Mem: 6.78 GB
0:	learn: 0.6551985	test: 0.6551377	best: 0.6551377 (0)	total: 114ms	remaining: 38m 5s
100:	learn: 0.1523731	test: 0.1521549	best: 0.1521549 (100)	total: 5.7s	remaining: 18m 43s
200:	learn: 0.1402271	test: 0.1407014	best: 0.1407014 (200)	total: 11s	remaining: 18m 5s
300:	learn: 0.1359149	test: 0.1371207	best: 0.1371207 (300)	total: 16.2s	remaining: 17m 39s
400:	learn: 0.1333834	test: 0.1352583	best: 0.1352583 (400)	total: 21.3s	remaining: 17m 22s
500:	learn: 0.1314935	test: 0.1341223	best: 0.1341223 (500)	total: 26.7s	remaining: 17m 18s
600:	learn: 0.1300176	test: 0.1333011	best: 0.1333011 (600)	total: 31.6s	remaining: 17m
700:	learn: 0.1282826	test: 0.1324485	best: 0.1324485 (700)	total: 37s	remaining: 16m 57s
800:	learn: 0.1259805	test: 0.1313574	best: 0.1313574 (800)	total: 42.8s	remaining: 17m 6s
900:	learn: 0.1232718	test: 0.1302014	best: 0.1302014 (900)	total: 48.9s	remaining: 17m 16s
1000:	learn: 0.1210228	test: 0.